### Convert annotations to the format used by ATLOP for finetuning.

Define the imports

In [2]:
import json
import re

In [3]:
import nltk
print('Nltk version: {}.'.format(nltk.__version__))

from nltk.tokenize import TreebankWordTokenizer as twt
from nltk.tokenize import WordPunctTokenizer as wpt
from nltk.tokenize.punkt import PunktSentenceTokenizer, PunktParameters
nltk.download('punkt_tab')

Nltk version: 3.9.4.


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/nothingtodo/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

Define paths to the annotation files

In [4]:
PATH_PLATINUM_TRAIN = "../Annotations/Train/platinum_quality/json_format/train_platinum.json"
PATH_GOLD_TRAIN = "../Annotations/Train/gold_quality/json_format/train_gold.json"
PATH_SILVER_TRAIN = "../Annotations/Train/silver_quality/json_format/train_silver.json"
PATH_BRONZE_TRAIN = "../Annotations/Train/bronze_quality/json_format/train_bronze.json"
PATH_DEV = "../Annotations/Dev/json_format/dev.json"

Define output paths

In [5]:
PATH_OUTPUT_PLATINUM_TRAIN = "../Train/RE/data/train_platinum.json"
PATH_OUTPUT_GOLD_TRAIN = "../Train/RE/data/train_gold.json"
PATH_OUTPUT_SILVER_TRAIN = "../Train/RE/data/train_silver.json"
PATH_OUTPUT_BRONZE_TRAIN = "../Train/RE/data/train_bronze.json"
PATH_OUTPUT_DEV = "../Train/RE/data/dev.json"

Load the input files into dictionary variables

In [6]:
#with open(PATH_PLATINUM_TRAIN, 'r', encoding='utf-8') as file:
#	train_platinum = json.load(file)

with open(PATH_GOLD_TRAIN, 'r', encoding='utf-8') as file:
	train_gold = json.load(file)

with open(PATH_SILVER_TRAIN, 'r', encoding='utf-8') as file:
	train_silver = json.load(file)
	
with open(PATH_BRONZE_TRAIN, 'r', encoding='utf-8') as file:
	train_bronze = json.load(file)

with open(PATH_DEV, 'r', encoding='utf-8') as file:
	dev = json.load(file)

Remove articles having less that 5 entities annotated

In [7]:
def get_articles_with_less_than_5_entities(data: dict):
    pmid_list = []
    for pmid, article in data.items():
        entities = article['entities']
        if len(entities) < 5:
            pmid_list.append(pmid)
    return pmid_list

def remove_articles_with_less_than_5_entities(data: dict, data_name: str):
    pmid_list = get_articles_with_less_than_5_entities(data)
    for pmid in pmid_list:
        del data[pmid]
    print(f'{data_name} - {len(pmid_list)} articles removed.')

In [8]:
#remove_articles_with_less_than_5_entities(train_platinum, "train_platinum")
remove_articles_with_less_than_5_entities(train_gold, "train_gold")
remove_articles_with_less_than_5_entities(train_silver, "train_silver")
remove_articles_with_less_than_5_entities(train_bronze, "train_bronze")
remove_articles_with_less_than_5_entities(dev, "dev")

train_gold - 1 articles removed.
train_silver - 4 articles removed.
train_bronze - 8 articles removed.
dev - 0 articles removed.


Remove articles without relations annotated

In [9]:
def remove_articles_without_relations(data: dict, data_name: str):
    pmid_list = []
    for pmid, article in data.items():
        if len(article['relations']) == 0:
            pmid_list.append(pmid)

    for pmid in pmid_list:
        del data[pmid]
    print(f'{data_name} - {len(pmid_list)} articles removed.')

In [10]:
#remove_articles_without_relations(train_platinum, "train_platinum")
remove_articles_without_relations(train_gold, "train_gold")
remove_articles_without_relations(train_silver, "train_silver")
remove_articles_without_relations(train_bronze, "train_bronze")
remove_articles_without_relations(dev, "dev")

train_gold - 42 articles removed.
train_silver - 13 articles removed.
train_bronze - 65 articles removed.
dev - 2 articles removed.


Tokenize the articles

In [20]:
# Define the tokens of length 2 that are not captured by the tokenizer
ILLEGAL_WORDS_2 = [').', '(<', '>)', '),', '.,', '].', '],', '.:', '>.', '>,', '))', '+)', '>-', '</', '[<', '-,', '.)', '™,']
# Define the tokens of length 3 that are not captured by the tokenizer
ILLEGAL_WORDS_3 = ['.),', '.].', '>),', '.).', '>).']

TRAILING_PUNCT = {',', '.', ';', ':', ')', ']', '}', '!', '?'}

def split_trailing_punctuation(tokens):
    new_tokens = []
    for tok, start, end in tokens:
        # split only if token has length > 1 and ends with punctuation
        if len(tok) > 1 and tok[-1] in TRAILING_PUNCT:
            new_tokens.append((tok[:-1], start, end - 1))
            new_tokens.append((tok[-1], end - 1, end))
        else:
            new_tokens.append((tok, start, end))
    return new_tokens

def tokenize_text(text):
    spans = list(wpt().span_tokenize(text))
    tokens = []

    for start, end in spans:
        word = text[start:end]
        if word in ILLEGAL_WORDS_2:
            word1 = text[start:end-1]
            word2 = text[end-1:end]
            tokens.append((word1, start, end-1))
            tokens.append((word2, end-1, end))
        elif word in ILLEGAL_WORDS_3:
            word1 = text[start:start+1]
            word2 = text[start+1:end-1]
            word3 = text[end-1:end]
            tokens.append((word1, start, start+1))
            tokens.append((word2, start+1, end-1))
            tokens.append((word3, end-1, end))
        else:
            tokens.append((word, start, end))

    tokens = split_trailing_punctuation(tokens)
    return tokens

def tokenize_docs(data: dict, data_name: str):
    print(f"Tokenizing articles in set {data_name}...")

    for pmid, article in data.items():
        title = article['metadata']['title']
        abstract = article['metadata']['abstract']

        article['tokenized_title'] = tokenize_text(title)
        article['tokenized_abstract'] = tokenize_text(abstract)

In [21]:
#tokenize_docs(train_platinum, "train_platinum")
tokenize_docs(train_gold, "train_gold")
tokenize_docs(train_silver, "train_silver")
tokenize_docs(train_bronze, "train_bronze")
tokenize_docs(dev, "dev")

Tokenizing articles in set train_gold...
Tokenizing articles in set train_silver...
Tokenizing articles in set train_bronze...
Tokenizing articles in set dev...


Adjust wrong annotations (i.e., annotations including partially annotated words) from the silver collection 

In [22]:
# Each PMIDs maps to a dict of {annotation_with_wrong_text_span: correct_text_span}
PARTIAL_WORDS = {
    '35275534': {
        (1395, 1412, 'abstract', 'Ruminococcusgnavus'): 'Ruminococcusgnavusgroup'  
    },
    '38963982': {
        (92, 94, 'title', 'TAM'): 'TAMs',
        (435, 451, 'abstract', 'Intestinal tissue'): 'Intestinal tissues'
    },
    '38959280': {
        (74, 76, 'abstract', 'SGM'): 'SGMs',
        (695, 697, 'abstract', 'SGM'): 'SGMs',
        (266, 287, 'abstract', 'cisgender heterosexual'): 'cisgender heterosexuals',
        (713, 734, 'abstract', 'cisgender-heterosexual'): 'cisgender-heterosexuals' 
    },
    '38968876': {
        (764, 770, 'abstract', 'patient'): 'patients'
    },
    '38892525': {
        (1397, 1407, 'abstract', 'IBS symptom'): 'IBS symptoms'
    }
}

def fix_wrong_annotations(data: dict, data_name: str):
    print(f'Fixing annotations in set {data_name}...')
    
    for pmid in list(data.keys()):
        pmid_str = str(pmid)
        if pmid_str in PARTIAL_WORDS:
            replacements = PARTIAL_WORDS[pmid_str]
            
            # Fix entities:
            for entity in data[pmid]['entities']:
                wrong_entry = (entity['start_idx'], entity['end_idx'], entity['location'], entity['text_span'])
                if wrong_entry in replacements:
                    correct_text = replacements[wrong_entry]
                    entity['text_span'] = correct_text
                    # Update end index based on the new text length.
                    entity['end_idx'] = entity['start_idx'] + len(correct_text) - 1
                    print(f"Fixed entity in pmid {pmid}: '{wrong_entry[3]}' -> '{correct_text}'")
            
            # Fix relations:
            for relation in data[pmid]['relations']:
                # Check and fix subject if needed.
                wrong_entry = (relation['subject_start_idx'], relation['subject_end_idx'], relation['subject_location'], relation['subject_text_span'])
                if wrong_entry in replacements:
                    correct_text = replacements[wrong_entry]
                    relation['subject_text_span'] = correct_text
                    relation['subject_end_idx'] = relation['subject_start_idx'] + len(correct_text) - 1
                    print(f"Fixed subject in pmid {pmid}: '{wrong_entry[3]}' -> '{correct_text}'")
                    
                # Check and fix object if needed.
                wrong_entry = (relation['object_start_idx'], relation['object_end_idx'], relation['object_location'], relation['object_text_span'])
                if wrong_entry in replacements:
                    correct_text = replacements[wrong_entry]
                    relation['object_text_span'] = correct_text
                    relation['object_end_idx'] = relation['object_start_idx'] + len(correct_text) - 1
                    print(f"Fixed object in pmid {pmid}: '{wrong_entry[3]}' -> '{correct_text}'")

In [23]:
fix_wrong_annotations(train_silver, 'train_silver')

Fixing annotations in set train_silver...


Map annotated entities to tokens 

In [26]:
def map_entities_to_tokens(data: dict, data_name: str):
    print(f"Mapping entities to tokens in set {data_name}...")

    for pmid, article in data.items():
        for entity in article['entities']:
            location = entity['location']
            start = entity['start_idx']
            end = entity['end_idx']

            start_token = None
            end_token = None

            if location == 'title':
                tokens = article['tokenized_title']
            elif location == 'abstract':
                tokens = article['tokenized_abstract']
            else:
                raise Exception(f'{pmid} - Unrecognized Location: {location}')

            # 1) strict exact matching
            for idx, token in enumerate(tokens):
                if start == token[1] and start_token is None:
                    start_token = idx
                if end == token[2] - 1 and end_token is None:
                    end_token = idx

            # 2) fallback: containment-based matching
            if start_token is None or end_token is None:
                overlapping = []
                for idx, token in enumerate(tokens):
                    tok_start = token[1]
                    tok_end_inclusive = token[2] - 1

                    # token overlaps entity span
                    if not (tok_end_inclusive < start or tok_start > end):
                        overlapping.append(idx)

                if overlapping:
                    if start_token is None:
                        start_token = overlapping[0]
                    if end_token is None:
                        end_token = overlapping[-1]

            if start_token is not None and end_token is not None:
                entity['start_token'] = start_token
                entity['end_token'] = end_token
            else:
                print("PMID:", pmid)
                print("ENTITY:", entity)
                print("TITLE TOKENS:", data[pmid]['tokenized_title'])
                print("ABSTRACT TOKENS:", data[pmid]['tokenized_abstract'])
                raise Exception(f'{pmid} - Not able to assign token(s) to entity: {entity}')

In [27]:
#map_entities_to_tokens(train_platinum, "train_platinum")

map_entities_to_tokens(train_gold, "train_gold")
map_entities_to_tokens(train_silver, "train_silver")
map_entities_to_tokens(train_bronze, "train_bronze")
map_entities_to_tokens(dev, "dev")

Mapping entities to tokens in set train_gold...
Mapping entities to tokens in set train_silver...
Mapping entities to tokens in set train_bronze...
Mapping entities to tokens in set dev...


Get the start and end indices for articles sentences

In [30]:
# Define abbreviations to not be splitted by the sentence tokenizer
extra_abbrevs = {'etc', 'etc.', 'etc.)', '<i>L', 'sp', 'subsp', '<i>A', '(<i>Hippophae rhamnoides</i> L.)'}
punkt_param = PunktParameters()
for abbr in extra_abbrevs:
	punkt_param.abbrev_types.add(abbr)
sentence_splitter = PunktSentenceTokenizer(punkt_param)

def get_sentence_spans(data: dict, data_name: str):
    print(f"Getting sentence spans in set {data_name}...")
    
    for pmid, article in data.items():
        title = article['metadata']['title']
        abstract = article['metadata']['abstract']

        # Convert the generator to a list so we can iterate it repeatedly.
        sentences = list(sentence_splitter.span_tokenize(abstract))

        # Prepare a list of booleans that will flag whether the sentence at a given index should be merged with the next one. 
        # A sentence is merged with the following one if an entity spans across them.
        # Initially, no merge is flagged.
        merge_next = [False] * len(sentences)
        
        # Process each entity in the article.
        for entity in article['entities']:
            location = entity['location']
            start = entity['start_idx']
            end = entity['end_idx']

            # For title entities, we do nothing regarding sentence spans.
            if location == 'title':
                if end > len(title):
                    raise Exception(f'{pmid} - Found title entity having illegal end index: {entity}')
                continue

            # Only process abstract entities.
            if location == 'abstract':
                start_sentence = None
                end_sentence = None

                # Iterate over the original sentence spans to determine in which sentences the entity start and end fall.
                for idx, s in enumerate(sentences):
                    # Using >= and <= to include boundaries.
                    if start >= s[0] and start <= s[1] and start_sentence is None:
                        start_sentence = idx
                        #print(f'Start sentence assigned: {idx}')
                    if end >= s[0] and end <= s[1] and end_sentence is None:
                        end_sentence = idx
                        #print(f'End sentence assigned: {idx}')

                if start_sentence is None:
                    raise Exception(f'{pmid} - Start sentence not assigned for entity: {entity}')
                if end_sentence is None:
                    raise Exception(f'{pmid} - End sentence not assigned for entity: {entity}')
                
                # If the entity falls in two different sentences, check if they are consecutive.
                if start_sentence != end_sentence:
                    # merge all intermediate consecutive sentences until the entity is fully covered
                    for i in range(start_sentence, end_sentence):
                        merge_next[i] = True

        # At this point, we have a merge flag for each sentence that should be merged with its next one.
        # Now we build the updated list of sentence spans, merging as flagged.
        new_spans = []
        i = 0
        while i < len(sentences):
            start_val = sentences[i][0]
            end_val = sentences[i][1]
            # While the current sentence is flagged to merge with the next one, update the end_val.
            while i < len(sentences) - 1 and merge_next[i]:
                i += 1
                end_val = sentences[i][1]
            new_spans.append((start_val, end_val))
            i += 1

        # Add the updated list of sentence spans to the article dictionary.
        article['sentences'] = new_spans

In [31]:
#get_sentence_spans(train_platinum, "train_platinum")
get_sentence_spans(train_gold, "train_gold")
get_sentence_spans(train_silver, "train_silver")
get_sentence_spans(train_bronze, "train_bronze")
get_sentence_spans(dev, "dev")

Getting sentence spans in set train_gold...
Getting sentence spans in set train_silver...
Getting sentence spans in set train_bronze...
Getting sentence spans in set dev...


Check if sentence spans have been computed correctly.

In [32]:
def check_sentence_spans(data: dict, data_name: str):
    print(f"Checking sentence spans in set {data_name}...")

    for pmid, article in data.items():
        # Process each entity in the article.
        for entity in article['entities']:
            location = entity['location']
            start = entity['start_idx']
            end = entity['end_idx']
            # For title entities, we do nothing regarding sentence spans.
            if location == 'title':
                continue

            # Only process abstract entities.
            if location == 'abstract':
                start_sentence = None
                end_sentence = None

                # Iterate over the original sentence spans to determine in which sentences the entity start and end fall.
                for idx, s in enumerate(article['sentences']):
                    # Using >= and <= to include boundaries.
                    if start >= s[0] and start <= s[1] and start_sentence is None:
                        start_sentence = idx
                        #print(f'Start sentence assigned: {idx}')
                    if end >= s[0] and end <= s[1] and end_sentence is None:
                        end_sentence = idx
                        #print(f'End sentence assigned: {idx}')

                if start_sentence is None:
                    raise Exception(f'{pmid} - Start sentence not assigned for entity: {entity}')
                if end_sentence is None:
                    raise Exception(f'{pmid} - End sentence not assigned for entity: {entity}')
                
                # If the entity falls in two different sentences, raise Exception.
                if start_sentence != end_sentence:
                      raise Exception(f'{pmid} - Entity assigned to two different sentences ({start_sentence}, {end_sentence}): {entity}')


In [33]:
#check_sentence_spans(train_platinum, "train_platinum")
check_sentence_spans(train_gold, "train_gold")
check_sentence_spans(train_silver, "train_silver")
check_sentence_spans(train_bronze, "train_bronze")
check_sentence_spans(dev, "dev")

Checking sentence spans in set train_gold...
Checking sentence spans in set train_silver...
Checking sentence spans in set train_bronze...
Checking sentence spans in set dev...


Map tokens to the sentence in which they are located

In [36]:
def map_tokens_to_sentences(data: dict, data_name: str):
    """
    For each article, map tokens in the 'tokenized abstract' to the sentence in which they are located.
    Uses the 'sentences' field in the article, which is assumed to be a list of (start, end) tuples.
    
    The mapping is stored as a dictionary where the key is the token index (its position in the tokenized abstract)
    and the value is the sentence index. For example, if the first token belongs to sentence 0 and the third token
    belongs to sentence 1, the mapping will include entries {0: 0, 2: 1}.
    
    Raises an Exception if a token does not fall within any of the sentence spans.
    """
    print(f"Mapping tokens to sentences in set {data_name}...")

    for pmid, article in data.items():
        # Retrieve the tokenized abstract and the sentence spans.
        tokens = article.get('tokenized_abstract')
        sentences = article.get('sentences')
        
        if tokens is None:
            raise Exception(f"Article {pmid} is missing 'tokenized abstract'.")
        if sentences is None:
            raise Exception(f"Article {pmid} is missing 'sentences'. Make sure to run get_sentence_spans first.")
        
        token_to_sentence = {}
        
        # Iterate over each token and determine which sentence it belongs to.
        for token_index, token_entry in enumerate(tokens):
            # Each token_entry is assumed to be a tuple: (token_text, start_offset, end_offset)
            token_text, token_start, token_end = token_entry
            assigned_sentence = None
            
            # Check each sentence span to see if the token falls within it.
            for sentence_index, (sent_start, sent_end) in enumerate(sentences):
                # allow overlap instead of strict containment
                if not (token_end <= sent_start or token_start >= sent_end):
                    assigned_sentence = sentence_index
                    break
            
            if assigned_sentence is None:
                raise Exception(
                    f"Token '{token_text}' (index {token_index}, offsets {token_start}-{token_end}) "
                    f"in article {pmid} does not fall within any sentence span: {sentences}"
                )
            
            token_to_sentence[token_index] = assigned_sentence
        
        # Add the mapping to the article dictionary.
        article['tokens_to_sentences_map'] = token_to_sentence

In [37]:
#map_tokens_to_sentences(train_platinum, "train_platinum")
map_tokens_to_sentences(train_gold, "train_gold")
map_tokens_to_sentences(train_silver, "train_silver")
map_tokens_to_sentences(train_bronze, "train_bronze")
map_tokens_to_sentences(dev, "dev")

Mapping tokens to sentences in set train_gold...
Mapping tokens to sentences in set train_silver...
Mapping tokens to sentences in set train_bronze...
Mapping tokens to sentences in set dev...


Map entities to the token positions within each sentence containing them

In [38]:
def map_entities_to_tokens_within_sentences(data: dict, data_name: str) -> dict:
    """
    For each article, this function maps each entity (assumed to be in the abstract)
    to the token positions within the sentence that contains it.
    
    For each entity in article['entities'] (with location 'abstract'), it adds:
      - 'located_in_sentence': the sentence index in which the entity's tokens are located,
      - 'start_token_in_sentence': the position of the entity's start token within that sentence,
      - 'end_token_in_sentence': the position of the entity's end token within that sentence.
    
    This function relies on:
      - article['tokenized abstract']: a list of tokens of the form (token_text, start_offset, end_offset)
      - article['tokens_to_sentences_map']: a mapping { token_index -> sentence_index }
      - article['sentences']: a list of (start, end) sentence spans for the abstract.
    """
    print(f"Mapping entities to tokens within sentences in set {data_name}...")
    
    for pmid, article in data.items():
        # Retrieve required fields.
        tokens = article.get('tokenized_abstract')
        token_to_sentence = article.get('tokens_to_sentences_map')
        sentences = article.get('sentences')
        
        if tokens is None:
            raise Exception(f"Article {pmid} is missing 'tokenized abstract'.")
        if token_to_sentence is None:
            raise Exception(f"Article {pmid} is missing 'tokens_to_sentences_map'. Run map_tokens_to_sentences first.")
        if sentences is None:
            raise Exception(f"Article {pmid} is missing 'sentences'. Run get_sentence_spans first.")
        
        # Build a helper mapping: for each sentence index, list the token indices that fall into that sentence.
        sentence_to_token_indices = {}
        for token_index in range(len(tokens)):
            sent_idx = token_to_sentence.get(token_index)
            if sent_idx is None:
                raise Exception(
                    f"In article {pmid}, token index {token_index} is not mapped to any sentence. Tokens: {tokens[token_index]}"
                )
            sentence_to_token_indices.setdefault(sent_idx, []).append(token_index)
        
        # Now process each entity.
        for entity in article.get('entities', []):
            if entity.get('location') != 'abstract': # Only process entities in the abstract.
                continue
            
            # Retrieve the token indices for this entity.
            entity_start_token = entity.get('start_token')
            entity_end_token = entity.get('end_token')
            
            if entity_start_token is None or entity_end_token is None:
                raise Exception(
                    f"Entity in article {pmid} is missing start_token or end_token: {entity}"
                )
            
            # Determine the sentence in which the entity's tokens are located.
            sentence_for_start = token_to_sentence.get(entity_start_token)
            sentence_for_end = token_to_sentence.get(entity_end_token)
            
            if sentence_for_start is None or sentence_for_end is None:
                raise Exception(
                    f"Entity in article {pmid} has tokens not mapped to any sentence: {entity}"
                )
            
            if sentence_for_start != sentence_for_end:
                raise Exception(
                    f"Entity in article {pmid} spans multiple sentences (start in {sentence_for_start}, end in {sentence_for_end}): {entity}"
                )
            
            located_sentence = sentence_for_start  # or sentence_for_end, both are same.
            
            # Get the list of token indices for the sentence.
            tokens_in_sentence = sentence_to_token_indices.get(located_sentence)
            if tokens_in_sentence is None:
                raise Exception(
                    f"Sentence {located_sentence} not found in helper mapping for article {pmid}."
                )
            
            # Find the position within the sentence for the start token.
            try:
                start_token_in_sentence = tokens_in_sentence.index(entity_start_token)
            except ValueError:
                raise Exception(
                    f"Entity start token {entity_start_token} not found in sentence tokens {tokens_in_sentence} for article {pmid}."
                )
            
            # And the position within the sentence for the end token.
            try:
                end_token_in_sentence = tokens_in_sentence.index(entity_end_token)
            except ValueError:
                raise Exception(
                    f"Entity end token {entity_end_token} not found in sentence tokens {tokens_in_sentence} for article {pmid}."
                )
            
            # Add the new fields to the entity.
            entity['located_in_sentence'] = located_sentence
            entity['start_token_in_sentence'] = start_token_in_sentence
            entity['end_token_in_sentence'] = end_token_in_sentence

In [39]:
#map_entities_to_tokens_within_sentences(train_platinum, "train_platinum")
map_entities_to_tokens_within_sentences(train_gold, "train_gold")
map_entities_to_tokens_within_sentences(train_silver, "train_silver")
map_entities_to_tokens_within_sentences(train_bronze, "train_bronze")
map_entities_to_tokens_within_sentences(dev, "dev")

Mapping entities to tokens within sentences in set train_gold...
Mapping entities to tokens within sentences in set train_silver...
Mapping entities to tokens within sentences in set train_bronze...
Mapping entities to tokens within sentences in set dev...


Map each relation's subject and object to an index corresponding to their position in the article's 'entities' list.

In [40]:
def map_relations_to_entities(data: dict, data_name: str) -> dict:
    """
    For each article, this function maps each relation's subject and object to an index 
    corresponding to their position in the article's 'entities' list. It adds two new fields 
    to each relation:
      - 'subject_entity_idx': the index of the subject entity in the article's 'entities' list.
      - 'object_entity_idx': the index of the object entity in the article's 'entities' list.
      
    The matching is performed based on:
      - For the subject:
            relation['subject_location'] == entity['location']
            relation['subject_start_idx'] == entity['start_idx']
            relation['subject_end_idx'] == entity['end_idx']
      - For the object:
            relation['object_location'] == entity['location']
            relation['object_start_idx'] == entity['start_idx']
            relation['object_end_idx'] == entity['end_idx']
            
    It is assumed that the article['entities'] list is ordered such that title entities come first (starting at index 0)
    and abstract entities follow (with indices continuing from the last title entity index).
    """
    print(f"Mapping relations to entities in set {data_name}...")
    
    for pmid, article in data.items():
        entities = article.get('entities')
        relations = article.get('relations')
        
        if entities is None:
            raise Exception(f"Article {pmid} is missing the 'entities' field.")
        if relations is None:
            # If there are no relations, there's nothing to map.
            continue
        
        # Process each relation.
        for relation in relations:
            # Map the subject.
            subject_idx_found = None
            for idx, entity in enumerate(entities):
                if (entity.get('location') == relation.get('subject_location') and
                    entity.get('start_idx') == relation.get('subject_start_idx') and
                    entity.get('end_idx') == relation.get('subject_end_idx')):
                    subject_idx_found = idx
                    break
                    
            if subject_idx_found is None:
                raise Exception(
                    f"Subject entity not found for relation in article {pmid}: {relation}"
                )
            relation['subject_entity_idx'] = subject_idx_found
            
            # Map the object.
            object_idx_found = None
            for idx, entity in enumerate(entities):
                if (entity.get('location') == relation.get('object_location') and
                    entity.get('start_idx') == relation.get('object_start_idx') and
                    entity.get('end_idx') == relation.get('object_end_idx')):
                    object_idx_found = idx
                    break
                    
            if object_idx_found is None:
                raise Exception(
                    f"Object entity not found for relation in article {pmid}: {relation}"
                )
            relation['object_entity_idx'] = object_idx_found

In [41]:
#map_relations_to_entities(train_platinum, "train_platinum")
map_relations_to_entities(train_gold, "train_gold")
map_relations_to_entities(train_silver, "train_silver")
map_relations_to_entities(train_bronze, "train_bronze")
map_relations_to_entities(dev, "dev")

Mapping relations to entities in set train_gold...
Mapping relations to entities in set train_silver...
Mapping relations to entities in set train_bronze...
Mapping relations to entities in set dev...


Convert processed annotations to the DocRED format used by ATLOP for finetuning

In [42]:
def convert_to_docred_format(data: dict, data_name: str, is_test=False) -> list:
    """
    Converts articles (in our intermediate format) to the DocRED format.
    
    For each article, a new dictionary is produced with the following keys:
      - "vertexSet": a list of entity mentions (each entity becomes a list with one mention).
          Each mention is a dict with:
              "pos": [start_token_in_sentence, end_token_in_sentence],
              "type": entity label,
              "sent_id": sentence id (0 for title; abstract sentences are numbered starting at 1),
              "name": the entity text span.
      - "labels": a list of relations, each a dict with:
              "r": relation predicate/label,
              "h": subject_entity_idx,
              "t": object_entity_idx,
              "evidence": a list of two sentence ids: [subject sentence id, object sentence id].
      - "title": the title string of the article.
      - "sents": a list of lists of tokens. The first entry is the title tokenization and subsequent
                 entries are the tokenizations of the abstract sentences.
    
    For abstract sentence tokenization, we use the sentence spans in article['sentences'] (a list of (start, end) offsets)
    and the tokenized abstract (article['tokenized abstract'], where each token is a tuple (token_text, start, end)).
    
    Returns a list of DocRED-formatted document dictionaries.
    """
    print(f"Converting articles to DocRED format for set {data_name}...")

    docred_docs = []
    for pmid, article in data.items():
        # 1. Build the vertexSet.
        # Each entity becomes a single mention. We assume that the entities in article['entities'] 
        # are ordered with title entities first, then abstract entities.
        vertexSet = []
        for entity in article.get('entities', []):
            # Determine the sentence id according to DocRED.
            # Title entities are assigned to sentence 0.
            if entity.get('location') == 'title':
                sent_id = 0
            else:
                # For abstract entities, we expect a field 'located_in_sentence' computed earlier.
                # In our intermediate format abstract sentences are numbered 0,1,... but in DocRED the title is sentence 0.
                # So add 1.
                sent_id = entity.get('located_in_sentence', 0) + 1

            # The token offsets of the entity within its sentence.
            if entity['location'] == 'title':
                pos = [entity.get('start_token'), entity.get('end_token')+1]
            else:
                pos = [entity.get('start_token_in_sentence'), entity.get('end_token_in_sentence')+1]
            mention = {
                "pos": pos,
                "type": entity.get("label").upper(),
                "sent_id": sent_id,
                "name": entity.get("text_span")
            }
            # Each vertexSet entry is a list of mentions (we have one per entity).
            vertexSet.append([mention])
        
        # 2. Build the labels (relations).
        labels = []
        for relation in article.get("relations", []):
            # Get the indices of the subject and object entities.
            subj_idx = relation.get("subject_entity_idx")
            obj_idx = relation.get("object_entity_idx")
            if subj_idx is None or obj_idx is None:
                raise Exception(f"Missing entity mapping in relation: {relation} in article {pmid}")
            subj_entity = article["entities"][subj_idx]
            obj_entity = article["entities"][obj_idx]
            
            # Determine the sentence id for each entity.
            if subj_entity.get("location") == "title":
                subj_sent = 0
            else:
                subj_sent = subj_entity.get("located_in_sentence", 0) + 1
                
            if obj_entity.get("location") == "title":
                obj_sent = 0
            else:
                obj_sent = obj_entity.get("located_in_sentence", 0) + 1
            
            relation_dict = {
                "r": relation.get("predicate").upper(),
                "h": subj_idx,
                "t": obj_idx,
                "evidence": [subj_sent, obj_sent] if subj_sent != obj_sent else [subj_sent]
            }
            labels.append(relation_dict)
        
        # 3. Build the sents field.
        # The first sentence is the tokenization of the title.
		
        title_tokens = article.get("tokenized_title", [])
        tokens_in_title = []
        for token in title_tokens:
            tokens_in_title.append(token[0])
        sents = [tokens_in_title]
            
        # For the abstract sentences, we use article['sentences'] (list of (start, end) spans)
        # and article['tokenized abstract'] (list of tokens, each as (token_text, start, end)).
        abstract_tokens = article.get("tokenized_abstract", [])
        abstract_sents = []
        for span in article.get("sentences", []):
            s_start, s_end = span
            tokens_in_sentence = []
            for token, t_start, t_end in abstract_tokens:
                # If a token falls completely within the sentence span, add it.
                if t_start >= s_start and t_end <= s_end:
                    tokens_in_sentence.append(token)
            abstract_sents.append(tokens_in_sentence)
        sents.extend(abstract_sents)
        
        # 4. The title string.
        doc_title = article["metadata"]["title"]
        
        # 5. Build the final document dictionary.
        if is_test:
            doc = {
                "vertexSet": vertexSet,
                "title": doc_title,
                "sents": sents
            }
        else:
            doc = {
                "vertexSet": vertexSet,
                "labels": labels,
                "title": doc_title,
                "sents": sents
            }
        docred_docs.append(doc)
    
    return docred_docs


In [43]:
#docred_train_platinum = convert_to_docred_format(train_platinum, "train_platinum")
docred_train_gold = convert_to_docred_format(train_gold, "train_gold")
docred_train_silver = convert_to_docred_format(train_silver, "train_silver")
docred_train_bronze = convert_to_docred_format(train_bronze, "train_bronze")
docred_dev = convert_to_docred_format(dev, "dev")

Converting articles to DocRED format for set train_gold...
Converting articles to DocRED format for set train_silver...
Converting articles to DocRED format for set train_bronze...
Converting articles to DocRED format for set dev...


Dump the dictionary variable to a json file

In [44]:
def dump_to_json(docred_dict, output_file_path):
	dict_with_double_quotes = json.dumps(docred_dict, ensure_ascii=False)
	with open(output_file_path, 'w', encoding='utf-8') as f:
		f.write(dict_with_double_quotes)

#dump_to_json(docred_train_platinum, PATH_OUTPUT_PLATINUM_TRAIN)
dump_to_json(docred_train_gold, PATH_OUTPUT_GOLD_TRAIN)
dump_to_json(docred_train_silver, PATH_OUTPUT_SILVER_TRAIN)
dump_to_json(docred_train_bronze, PATH_OUTPUT_BRONZE_TRAIN)
dump_to_json(docred_dev, PATH_OUTPUT_DEV)